# 01 · Profiling the raw DE-SynPUF data (Sample 2)

**Goal:** understand what is actually in each file *before* writing cleaning rules.
Profiling answers four questions:
1. **Grain:** what does one row represent, and which columns uniquely identify it?
2. **Completeness:** which columns are empty, mostly empty, or useless?
3. **Validity:** are dates real dates, are costs plausible, are codes real codes?
4. **Linkage:** does every claim belong to a known member?

**What this notebook does NOT do:** change data. It opens the database read-only.
Every finding here becomes a rule in Phase 2 (`src/validate.py`), listed at the bottom.

**To run:** activate `.venv`, run `jupyter notebook`, open this file, then *Run → Run All Cells*.
Requires `python src/download.py` and `python src/ingest.py` to have been run first.

In [23]:
import os, sys
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from profiling import profile_table, check_key, top_values

# CLAIMS_DB lets you point at a different database without editing code.
DB = os.environ.get("CLAIMS_DB", str(ROOT / "data" / "processed" / "claims.duckdb"))
con = duckdb.connect(DB, read_only=True)   # read-only: profiling must never modify data
pd.set_option("display.max_rows", 100)

q = lambda sql: con.execute(sql).df()       # tiny helper: SQL in, DataFrame out
print("DuckDB", duckdb.__version__, "|", Path(DB).name)

DuckDB 2.0.0 | claims.duckdb


## 1. What got loaded

`raw._load_log` is written by `ingest.py` on every run. One row per table per run, so re-runs show up as repeated rows. That's a feature: it's a history.

In [2]:
q("SELECT * FROM raw._load_log ORDER BY loaded_at DESC, table_name")

,table_name,files_loaded,row_count,loaded_at
0,pde,1,5561154,2026-10-27 12:11:40.145201
1,outpatient,1,792562,2026-10-27 12:11:35.501770
2,inpatient,1,66494,2026-10-27 12:11:30.687177
3,beneficiary,3,343858,2026-10-27 12:11:29.869200


Members per year. The beneficiary data is **one row per member per year**, so it has three rows for most people.
Counts shrink each year because members die or leave Medicare fee-for-service, and nobody new joins.

In [3]:
q("""
SELECT file_year, COUNT(*) AS members,
       COUNT(*) FILTER (WHERE BENE_DEATH_DT IS NOT NULL) AS died_that_year
FROM raw.beneficiary GROUP BY 1 ORDER BY 1
""")

,file_year,members,died_that_year
0,2008,116395,1777
1,2009,114618,1773
2,2010,112845,1818


## 2. Grain: what identifies a row?

A **key** is the set of columns that uniquely identifies a row. Getting this wrong is the most common way to double-count money.
`check_key` compares total rows with exact distinct key values. If they differ, it isn't a key.

In [4]:
pd.DataFrame([
    check_key(con, "raw.beneficiary", ["DESYNPUF_ID"]),
    check_key(con, "raw.beneficiary", ["DESYNPUF_ID", "file_year"]),
    check_key(con, "raw.inpatient",   ["CLM_ID"]),
    check_key(con, "raw.inpatient",   ["CLM_ID", "SEGMENT"]),
    check_key(con, "raw.outpatient",  ["CLM_ID"]),
    check_key(con, "raw.outpatient",  ["CLM_ID", "SEGMENT"]),
    check_key(con, "raw.pde",         ["PDE_ID"]),
])

,table,key,rows,distinct,duplicates,is_unique
0,raw.beneficiary,DESYNPUF_ID,343858,116395,227463,False
1,raw.beneficiary,"DESYNPUF_ID, file_year",343858,343858,0,True
2,raw.inpatient,CLM_ID,66494,66434,60,False
3,raw.inpatient,"CLM_ID, SEGMENT",66494,66494,0,True
4,raw.outpatient,CLM_ID,792562,781553,11009,False
5,raw.outpatient,"CLM_ID, SEGMENT",792562,792562,0,True
6,raw.pde,PDE_ID,5561154,5561154,0,True


**Finding: `CLM_ID` alone is not unique.** Some claims have a second row with `SEGMENT = 2`, so the claim key is `(CLM_ID, SEGMENT)`.

What's in a segment-2 row?

In [5]:
q("""
SELECT 'inpatient' AS file, SEGMENT, COUNT(*) AS rows,
       COUNT(CLM_FROM_DT) AS rows_with_dates,
       COUNT(*) FILTER (WHERE TRY_CAST(CLM_PMT_AMT AS DOUBLE) <> 0) AS rows_with_payment
FROM raw.inpatient GROUP BY ALL
UNION ALL
SELECT 'outpatient', SEGMENT, COUNT(*), COUNT(CLM_FROM_DT),
       COUNT(*) FILTER (WHERE TRY_CAST(CLM_PMT_AMT AS DOUBLE) <> 0)
FROM raw.outpatient GROUP BY ALL
ORDER BY 1, 2
""")

,file,SEGMENT,rows,rows_with_dates,rows_with_payment
0,inpatient,1,66434,66434,64455
1,inpatient,2,60,0,59
2,outpatient,1,781361,781361,751430
3,outpatient,2,11201,0,10968


In [6]:
# One example claim, both segments side by side
q("""
SELECT CLM_ID, SEGMENT, CLM_FROM_DT, CLM_THRU_DT, CLM_PMT_AMT, ICD9_DGNS_CD_1
FROM raw.outpatient
WHERE CLM_ID = (SELECT MIN(CLM_ID) FROM raw.outpatient WHERE SEGMENT = '2')
ORDER BY SEGMENT
""")

,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,CLM_PMT_AMT,ICD9_DGNS_CD_1
0,391012253926309,1,20090425,20090515,3300.00,438
1,391012253926309,2,NaN,NaN,0.00,OTHER


**Interpretation:**
- Segment-2 rows are **continuations** of a claim. They have no dates of their own, but most carry a **payment**.
- So "missing dates" here is structural, not an error. The fix is to inherit dates from segment 1.
- Dropping segment-2 rows would understate cost. Counting them as separate claims would overstate claim volume.
- **Phase 2 rule:** count claims as `DISTINCT CLM_ID`; sum payments across all segments; fill segment-2 dates from segment 1.

## 3. Completeness: which columns are empty or useless?

`profile_table` wraps DuckDB's `SUMMARIZE`, which profiles every column in a single scan.

In [7]:
profiles = {t: profile_table(con, f"raw.{t}") for t in ["beneficiary", "inpatient", "outpatient", "pde"]}

pd.DataFrame([{
    "table": t,
    "rows": p.attrs["n_rows"],
    "columns": len(p),
    "all_null_cols": int(p.all_null.sum()),
    "over_90pct_null": int((p.pct_null > 90).sum()),
    "constant_cols": ", ".join(p.loc[p.constant, "column"]) or "-",
} for t, p in profiles.items()])

,table,rows,columns,all_null_cols,over_90pct_null,constant_cols
0,beneficiary,343858,34,0,1,-
1,inpatient,66494,82,45,48,source_file
2,outpatient,792562,77,3,44,source_file
3,pde,5561154,9,0,0,source_file


In [8]:
# Inpatient: which columns are completely empty?
ip = profiles["inpatient"]
print(", ".join(ip.loc[ip.all_null, "column"]))

HCPCS_CD_1, HCPCS_CD_2, HCPCS_CD_3, HCPCS_CD_4, HCPCS_CD_5, HCPCS_CD_6, HCPCS_CD_7, HCPCS_CD_8, HCPCS_CD_9, HCPCS_CD_10, HCPCS_CD_11, HCPCS_CD_12, HCPCS_CD_13, HCPCS_CD_14, HCPCS_CD_15, HCPCS_CD_16, HCPCS_CD_17, HCPCS_CD_18, HCPCS_CD_19, HCPCS_CD_20, HCPCS_CD_21, HCPCS_CD_22, HCPCS_CD_23, HCPCS_CD_24, HCPCS_CD_25, HCPCS_CD_26, HCPCS_CD_27, HCPCS_CD_28, HCPCS_CD_29, HCPCS_CD_30, HCPCS_CD_31, HCPCS_CD_32, HCPCS_CD_33, HCPCS_CD_34, HCPCS_CD_35, HCPCS_CD_36, HCPCS_CD_37, HCPCS_CD_38, HCPCS_CD_39, HCPCS_CD_40, HCPCS_CD_41, HCPCS_CD_42, HCPCS_CD_43, HCPCS_CD_44, HCPCS_CD_45


**Finding:** all 45 `HCPCS_CD_*` columns in inpatient are 100% empty. Inpatient stays are billed by DRG (`CLM_DRG_CD`), not HCPCS. **Phase 2 rule:** drop them from the clean table.

`constant_cols` shows only `source_file`, which is expected for single-file tables.

In [9]:
profiles["pde"]

,column,pct_null,approx_distinct,min,max,all_null,constant,id_like
0,DESYNPUF_ID,0.0,103654,00000B48BCF4AD29,FFFF52C0C450A585,False,False,False
1,PDE_ID,0.0,5529509,83014461672923,83994467362454,False,False,True
2,SRVC_DT,0.0,1118,20080101,20101231,False,False,False
3,PROD_SRVC_ID,0.0,266726,00002000001,OTHER,False,False,False
4,QTY_DSPNSD_NUM,0.0,37,0.000,90.000,False,False,False
5,DAYS_SUPLY_NUM,0.0,10,0,90,False,False,False
6,PTNT_PAY_AMT,0.0,18,0.00,90.00,False,False,False
7,TOT_RX_CST_AMT,0.0,59,0.00,90.00,False,False,False
8,source_file,0.0,1,DE1_0_2008_to_2010_Prescription_Drug_Events_Sa...,DE1_0_2008_to_2010_Prescription_Drug_Events_Sa...,False,True,False


### A lesson about approximate counts

`SUMMARIZE` reports `approx_unique`, an *estimate* (HyperLogLog algorithm) that trades accuracy for speed.
Compare it with the exact answer:

In [10]:
approx = profiles["pde"].set_index("column").at["DESYNPUF_ID", "approx_distinct"]
exact  = q("SELECT COUNT(DISTINCT DESYNPUF_ID) AS n FROM raw.pde").n[0]
print(f"approx: {approx:,}   exact: {exact:,}   error: {100*(approx-exact)/exact:+.1f}%")

approx: 103,654   exact: 99,746   error: +3.9%


Approximations are fine for "roughly how big is this column?". They are **never** fine for a number that goes in a report.
Every count in the Phase 2 data-quality report uses exact `COUNT(DISTINCT ...)`.

## 4. Dates: valid format, valid logic

Dates arrive as `YYYYMMDD` text. There are two kinds of check:
- **Format:** is it 8 digits, and a real calendar date? (`20081399` would fail.)
- **Logic:** does it make sense relative to other dates? (Discharge before admission? Service after death?)

In [11]:
checks = [("inpatient","CLM_FROM_DT"),("inpatient","CLM_THRU_DT"),("inpatient","CLM_ADMSN_DT"),
          ("inpatient","NCH_BENE_DSCHRG_DT"),("outpatient","CLM_FROM_DT"),("outpatient","CLM_THRU_DT"),
          ("pde","SRVC_DT"),("beneficiary","BENE_BIRTH_DT"),("beneficiary","BENE_DEATH_DT")]
q(" UNION ALL ".join(f"""
    SELECT '{t}' AS tbl, '{c}' AS col,
           COUNT(*) FILTER (WHERE {c} IS NULL) AS n_null,
           COUNT(*) FILTER (WHERE {c} IS NOT NULL AND TRY_STRPTIME({c}, '%Y%m%d') IS NULL) AS not_a_date,
           MIN({c}) AS earliest, MAX({c}) AS latest
    FROM raw.{t}""" for t, c in checks))

,tbl,col,n_null,not_a_date,earliest,latest
0,inpatient,CLM_FROM_DT,60,0,20071127,20101229
1,inpatient,CLM_THRU_DT,60,0,20080101,20101231
2,inpatient,CLM_ADMSN_DT,0,0,20071127,20101229
3,inpatient,NCH_BENE_DSCHRG_DT,0,0,20080101,20101231
4,outpatient,CLM_FROM_DT,11201,0,20071212,20101231
5,outpatient,CLM_THRU_DT,11201,0,20080101,20101231
6,pde,SRVC_DT,0,0,20080101,20101231
7,beneficiary,BENE_BIRTH_DT,0,0,19090101,19831201
8,beneficiary,BENE_DEATH_DT,338490,0,20080101,20101201


In [12]:
q("""
WITH life AS (SELECT DESYNPUF_ID, MAX(BENE_DEATH_DT) AS death_dt FROM raw.beneficiary GROUP BY 1)
SELECT 'inpatient: thru before from'       AS check_name, COUNT(*) AS violations FROM raw.inpatient  WHERE CLM_THRU_DT < CLM_FROM_DT
UNION ALL SELECT 'inpatient: discharge before admit', COUNT(*) FROM raw.inpatient  WHERE NCH_BENE_DSCHRG_DT < CLM_ADMSN_DT
UNION ALL SELECT 'outpatient: thru before from',      COUNT(*) FROM raw.outpatient WHERE CLM_THRU_DT < CLM_FROM_DT
UNION ALL SELECT 'inpatient: service after death',    COUNT(*) FROM raw.inpatient  JOIN life USING (DESYNPUF_ID) WHERE CLM_FROM_DT > death_dt
UNION ALL SELECT 'outpatient: service after death',   COUNT(*) FROM raw.outpatient JOIN life USING (DESYNPUF_ID) WHERE CLM_FROM_DT > death_dt
UNION ALL SELECT 'rx: fill after death',              COUNT(*) FROM raw.pde        JOIN life USING (DESYNPUF_ID) WHERE SRVC_DT > death_dt
UNION ALL SELECT 'claims starting before 2008',       COUNT(*) FROM raw.inpatient  WHERE CLM_FROM_DT < '20080101'
""")

,check_name,violations
0,inpatient: thru before from,0
1,inpatient: discharge before admit,0
2,outpatient: thru before from,0
3,inpatient: service after death,0
4,outpatient: service after death,0
5,rx: fill after death,0
6,claims starting before 2008,204


**Finding:** no format or logic violations. CMS's synthesis kept dates internally consistent.

The nulls are exactly the segment-2 rows from Section 2. A few hundred claims **start in late 2007 and end in 2008**; they're stays crossing the year boundary.
**Phase 2 decision:** assign a claim to a year by its **through (end) date**. That's the CMS convention, and it keeps every claim inside 2008–2010.

The checks still go into Phase 2. A check that passes with 0 violations is evidence, and it would catch a problem if the data changed.

## 5. Linkage: does every claim belong to a known member?

An **orphaned key** is a claim whose `DESYNPUF_ID` has no matching member. There are two levels:
- **Ever:** the member exists in *any* year.
- **Same year:** the member has a beneficiary row *for the year of the claim*. This matters because demographics and chronic conditions are joined per year.

In [13]:
q("""
WITH b AS (SELECT DISTINCT DESYNPUF_ID, file_year FROM raw.beneficiary)
SELECT 'inpatient' AS tbl,
       COUNT(*) FILTER (WHERE DESYNPUF_ID NOT IN (SELECT DESYNPUF_ID FROM b)) AS orphan_ever,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.DESYNPUF_ID = c.DESYNPUF_ID
                                          AND b.file_year = CAST(LEFT(c.CLM_THRU_DT, 4) AS INT))) AS orphan_same_year
FROM raw.inpatient c WHERE SEGMENT = '1'
UNION ALL
SELECT 'outpatient',
       COUNT(*) FILTER (WHERE DESYNPUF_ID NOT IN (SELECT DESYNPUF_ID FROM b)),
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.DESYNPUF_ID = c.DESYNPUF_ID
                                          AND b.file_year = CAST(LEFT(c.CLM_THRU_DT, 4) AS INT)))
FROM raw.outpatient c WHERE SEGMENT = '1'
UNION ALL
SELECT 'pde',
       COUNT(*) FILTER (WHERE DESYNPUF_ID NOT IN (SELECT DESYNPUF_ID FROM b)),
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.DESYNPUF_ID = c.DESYNPUF_ID
                                          AND b.file_year = CAST(LEFT(c.SRVC_DT, 4) AS INT)))
FROM raw.pde c
""")

,tbl,orphan_ever,orphan_same_year
0,inpatient,0,0
1,outpatient,0,0
2,pde,0,0


**Finding:** zero orphans at both levels. This is the payoff of DECISIONS.md #4 and #8: mixing in another sample's beneficiary file would have made these numbers large.

## 6. Costs: negatives, zeros, and caps

Payment fields: `CLM_PMT_AMT` (what Medicare paid on a claim) and, for drugs, `TOT_RX_CST_AMT` (total drug cost) and `PTNT_PAY_AMT` (patient's share).

In [14]:
cost_cols = [("inpatient","CLM_PMT_AMT"),("outpatient","CLM_PMT_AMT"),("pde","TOT_RX_CST_AMT"),("pde","PTNT_PAY_AMT")]
q(" UNION ALL ".join(f"""
    SELECT '{t}' AS tbl, '{c}' AS col,
           COUNT(*) FILTER (WHERE v < 0)  AS negative,
           COUNT(*) FILTER (WHERE v = 0)  AS zero,
           QUANTILE_CONT(v, 0.50)         AS p50,
           QUANTILE_CONT(v, 0.99)         AS p99,
           MAX(v)                         AS max,
           COUNT(*) FILTER (WHERE v = (SELECT MAX(TRY_CAST({c} AS DOUBLE)) FROM raw.{t})) AS rows_at_max
    FROM (SELECT TRY_CAST({c} AS DOUBLE) AS v FROM raw.{t})""" for t, c in cost_cols))

,tbl,col,negative,zero,p50,p99,max,rows_at_max
0,inpatient,CLM_PMT_AMT,58,1980,7000.0,55000.0,57000.0,622
1,outpatient,CLM_PMT_AMT,2557,30164,80.0,3300.0,3300.0,8611
2,pde,TOT_RX_CST_AMT,0,550818,20.0,550.0,570.0,52874
3,pde,PTNT_PAY_AMT,0,3418739,0.0,140.0,170.0,43669


In [15]:
# Are amounts suspiciously round? Share of outpatient payments that are exact multiples of $10
q("""
SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(CLM_PMT_AMT AS DOUBLE) % 10 = 0) / COUNT(*), 1) AS pct_multiple_of_10
FROM raw.outpatient
""")

,pct_multiple_of_10
0,100.0


**Findings:**
- **Negative payments.** In real claims, negatives are usually *adjustments or reversals* of earlier payments. Deleting them would overstate cost.
  **Phase 2 rule:** keep, flag as `is_adjustment`, include in totals, and exclude from "typical claim" averages.
- **Many zero-payment claims**, for example claims denied or paid entirely by another payer. Keep and flag.
- **Rounded and capped amounts.** Every outpatient payment is a multiple of $10, and thousands of rows sit at *exactly* the maximum (see `rows_at_max`). CMS **top-coded** extreme values to protect privacy.
  Consequence: the high-cost-member analysis will understate the real long tail, and a z-score outlier test will mostly re-discover the cap. Say this in the summary.

## 7. Codes: placeholders and inconsistent coding

In [16]:
q("""
SELECT 'outpatient dx1 = OTHER' AS check_name, COUNT(*) AS n FROM raw.outpatient WHERE ICD9_DGNS_CD_1 = 'OTHER'
UNION ALL SELECT '  ...of which segment 2', COUNT(*) FROM raw.outpatient WHERE ICD9_DGNS_CD_1 = 'OTHER' AND SEGMENT = '2'
UNION ALL SELECT 'inpatient dx1 = OTHER', COUNT(*) FROM raw.inpatient WHERE ICD9_DGNS_CD_1 = 'OTHER'
UNION ALL SELECT 'rx drug code = OTHER', COUNT(*) FROM raw.pde WHERE PROD_SRVC_ID = 'OTHER'
""")

,check_name,n
0,outpatient dx1 = OTHER,5640
1,...of which segment 2,5616
2,inpatient dx1 = OTHER,0
3,rx drug code = OTHER,15


In [17]:
top_values(con, "raw.beneficiary", "BENE_ESRD_IND", 5)

,value,n,pct
0,0,316729,92.11
1,Y,27129,7.89


**Findings:**
- `OTHER` appears where a real ICD-9 code or drug code (NDC) should be. It's a synthesis placeholder.
  **Phase 2 rule:** convert it to NULL plus a flag, and exclude it from "top diagnoses."
- `BENE_ESRD_IND` (end-stage renal disease) is coded **`0` / `Y`**, not `N` / `Y` or `0` / `1`. **Phase 2 rule:** standardize it to a boolean.
- The chronic-condition flags (`SP_*`) take values **1 and 2**, not 0/1. The cell below tests which one means "yes."

In [18]:
# If 1 = yes, this is the share of members with diabetes each year
q("""
SELECT file_year,
       ROUND(100.0 * COUNT(*) FILTER (WHERE SP_DIABETES = '1') / COUNT(*), 1) AS pct_if_1_means_yes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE SP_DIABETES = '2') / COUNT(*), 1) AS pct_if_2_means_yes
FROM raw.beneficiary GROUP BY 1 ORDER BY 1
""")

,file_year,pct_if_1_means_yes,pct_if_2_means_yes
0,2008,37.9,62.1
1,2009,41.5,58.5
2,2010,29.1,70.9


Reading 1 = yes gives diabetes prevalence around 30–40%, which is in the plausible range for an older Medicare population. Reading 2 = yes would mean 60–70%, which isn't.
So **1 = yes, 2 = no is the likely coding**. This is an *inference from the data*, so **confirm it in the DE 1.0 Codebook** (linked on the CMS download page) before Phase 2 relies on it.
Using data to sanity-check documentation, and documentation to check data, is a habit worth naming in interviews.

In [19]:
# Top primary diagnoses on outpatient claims (segment 1, placeholder excluded)
top = q("""
SELECT ICD9_DGNS_CD_1 AS icd9, COUNT(*) AS claims
FROM raw.outpatient
WHERE SEGMENT = '1' AND ICD9_DGNS_CD_1 IS NOT NULL AND ICD9_DGNS_CD_1 <> 'OTHER'
GROUP BY 1 ORDER BY 2 DESC LIMIT 8
""")
labels = {"4019": "Essential hypertension, unspecified", "4011": "Essential hypertension, benign",
          "2720": "Pure hypercholesterolemia", "2721": "Pure hyperglyceridemia",
          "2722": "Mixed hyperlipidemia", "2723": "Hyperchylomicronemia",
          "2724": "Other/unspecified hyperlipidemia", "78651": "Precordial (chest) pain"}
top["description"] = top.icd9.map(labels).fillna("(look up)")
top

,icd9,claims,description
0,4019,18983,"Essential hypertension, unspecified"
1,4011,18717,"Essential hypertension, benign"
2,2723,4811,Hyperchylomicronemia
3,2721,4725,Pure hyperglyceridemia
4,2722,4700,Mixed hyperlipidemia
5,2724,4693,Other/unspecified hyperlipidemia
6,2720,4660,Pure hypercholesterolemia
7,78651,3419,Precordial (chest) pain


ICD-9 codes are stored **without the decimal point** (`4019` means `401.9`). Descriptions were added by hand for this preview.
Phase 3 will join a proper ICD-9 lookup table.

These results are plausible for an older Medicare population. But the even counts across the five 272.x lipid codes look **synthetic**:
real data wouldn't spread that evenly. The same goes for "unspecified" vs "benign" hypertension having nearly identical counts. That's another reminder that findings show method, not clinical truth.

## 8. Prescription logic checks

In [20]:
q("""
SELECT COUNT(*) AS fills,
       COUNT(*) FILTER (WHERE TRY_CAST(QTY_DSPNSD_NUM AS DOUBLE) = 0) AS qty_zero,
       COUNT(*) FILTER (WHERE TRY_CAST(DAYS_SUPLY_NUM AS INT) = 0)    AS days_supply_zero,
       MAX(TRY_CAST(DAYS_SUPLY_NUM AS INT))                            AS max_days_supply,
       COUNT(*) FILTER (WHERE TRY_CAST(PTNT_PAY_AMT AS DOUBLE) > TRY_CAST(TOT_RX_CST_AMT AS DOUBLE)) AS patient_paid_more_than_total
FROM raw.pde
""")

,fills,qty_zero,days_supply_zero,max_days_supply,patient_paid_more_than_total
0,5561154,232166,118118,90,288824


**Finding:** about 5% of fills say the patient paid **more than the total cost of the drug**. That's logically impossible: the patient's share is part of the total.
Zero quantity or zero days supply means the fill didn't really dispense anything.
These are the clearest **true validation failures** in the data.

**Phase 2 rule:** flag these fills (don't silently delete them), report the counts, and exclude flagged fills from cost-per-fill metrics.

## 9. Summary: findings → Phase 2 rules

| # | Finding | Severity | Phase 2 rule |
|---|---|---|---|
| 1 | Claim key is `(CLM_ID, SEGMENT)`, not `CLM_ID` | High (double-counting risk) | Count claims by `DISTINCT CLM_ID`; sum payments across segments |
| 2 | Segment-2 rows have no dates but carry payments | High | Inherit dates from segment 1; never drop segment 2 |
| 3 | 45 inpatient `HCPCS_CD_*` columns 100% empty | Low | Drop from clean table |
| 4 | Claims spanning 2007→2008 | Low | Assign claim year by through date |
| 5 | Negative payments | Medium | Keep, flag `is_adjustment`, exclude from averages |
| 6 | Many zero-payment claims | Low | Keep, flag |
| 7 | Costs rounded and top-coded | Medium (analysis caveat) | Document; use percentiles, not z-scores, for outliers |
| 8 | `OTHER` placeholder in dx and drug codes | Medium | NULL + flag; exclude from top-N |
| 9 | `BENE_ESRD_IND` coded `0`/`Y`; `SP_*` coded 1/2 (likely 1=yes, confirm in codebook) | Medium (silent misreads) | Standardize to booleans |
| 10 | Rx: patient paid > total; qty or days supply = 0 | High (impossible values) | Flag, report counts, exclude from cost-per-fill |
| 11 | Format, logic, and orphan checks all pass | Info | Keep the checks; zero violations is evidence |
| 12 | Approximate distinct counts can be far off | Process | Exact counts in all reported numbers |

In [28]:
q("""
SELECT 'inpatient' AS file,
    SUM(TRY_CAST(CLM_PMT_AMT AS DOUBLE)) FILTER (WHERE SEGMENT = '1')   AS seg1_only,
    SUM(TRY_CAST(CLM_PMT_AMT AS DOUBLE))                                AS all_segments
FROM raw.inpatient
UNION ALL
SELECT 'outpatient',
    SUM(TRY_CAST(CLM_PMT_AMT AS DOUBLE)) FILTER (WHERE SEGMENT = '1'),
    SUM(TRY_CAST(CLM_PMT_AMT AS DOUBLE))
FROM raw.outpatient
""")

,file,seg1_only,all_segments
0,inpatient,632653230.0,634063230.0
1,outpatient,210427890.0,225486960.0


In [21]:
con.close()